In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip -o "/content/drive/MyDrive/Bachelor's Project/data_27_cases_30_minutes.zip" -d /content
!unzip -o "/content/drive/MyDrive/Bachelor's Project/data_33_cases_30_minutes.zip" -d /content

In [ ]:
c = "All"
t_batch = 2.5

# Libraries and Modules

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from tensorflow.keras.layers import Input, Conv1D, Conv2D, MaxPool1D, MaxPool2D, Dense, Activation, Flatten, BatchNormalization, Dropout
from tensorflow.keras import Model
import keras.models
from keras import callbacks

# Dataset & Preprocessing

In [ ]:
data1 = np.load('/content/X.npy')
data2 = np.load('/content/data_33_cases_30_minutes.npy')

print(data1.shape)
print(data2.shape)

In [ ]:
def make_data(data, c, mci, t_batch, fs=256):

    N = int(fs * t_batch)          # Number of Samples in each *t_batch* seconds
    n_samples  = data.shape[0]     # Number of Samples
    n_channels = data.shape[1]     # Number of Channels
    n_batch = int(n_samples / N)

    x = []

    if c != 'All':

        for i in range(n_batch):
            var = data[i * N : (i+1) * N, c]    # Selecting c-th channel
            x.append(var)

    else:
        channel = data[:, :]

        for i in range(n_batch):
            var = channel[i * N : (i+1) * N, :]
            x.append(var)

    x = np.array(x)

    if mci:
        y = np.ones(x.shape[0], dtype='int')
    else:
        y = np.zeros(x.shape[0], dtype='int')

    return x, y


def make_dataset(data, mci_indices, c, t_batch, fs=256):

    X = []
    Y = []

    for i in range(len(data)):
        mci = True if i in mci_indices else False
        x, y = make_data(data[i], c=c, mci=mci, fs=fs, t_batch=t_batch)
        X.append(x)
        Y.append(y)

    X = np.concatenate(X, axis=0)
    Y = np.hstack(Y)

    return X, Y


In [ ]:
mci_indices_1 = [1, 2, 5, 6, 8, 10, 13, 14, 19, 22, 24]
mci_indices_2 = [4, 5, 7, 11, 12, 15, 18, 19, 20, 21, 22, 23, 26, 27, 29, 30, 31, 32]

X1, Y1 = make_dataset(data1, mci_indices_1, c=c, t_batch=t_batch)
X2, Y2 = make_dataset(data2, mci_indices_2, c=c, t_batch=t_batch)

if c != 'All':
    X1 = tf.expand_dims(X1, axis=-1)
    X2 = tf.expand_dims(X2, axis=-1)

print(X1.shape)
print(Y1.shape)

print(X2.shape)
print(Y2.shape)

In [ ]:
X = np.concatenate((X1, X2))
Y = np.concatenate((Y1, Y2))

print(X.shape)
print(Y.shape)

del X1, X2, Y1, Y2

In [ ]:
type(X)

# Building ConvNet

In [ ]:
## 2.5 seconds
def build_model():

    inputs = Input(shape=(int(t_batch * 256), 19))

    X = tf.keras.layers.LayerNormalization()(inputs)

    X = Conv1D(kernel_size=7, filters=16, activation='relu')(X)
    X = Dropout(rate = 0.2)(X)
    X = BatchNormalization()(X)
    X = MaxPool1D(2, 2)(X)

    X = Conv1D(kernel_size=13, filters=32, activation='relu')(X)
    X = Dropout(rate = 0.2)(X)
    X = BatchNormalization()(X)
    X = MaxPool1D(2, 2)(X)

    X = Conv1D(kernel_size=19, filters=64, activation='relu')(X)
    X = Dropout(rate = 0.2)(X)
    X = BatchNormalization()(X)
    X = MaxPool1D(2, 2)(X)

    X = Flatten()(X)

    X = Dense(100, activation='relu')(X)
    outputs = Dense(units = 1, name = 'sigmoid', activation='sigmoid')(X)

    model = Model(inputs=inputs, outputs=outputs)

    return model

## 5 seconds
# def build_model():

#     inputs = Input(shape=(int(t_batch * 256), 19))

#     X = tf.keras.layers.LayerNormalization()(inputs)

#     X = Conv1D(kernel_size=9, filters=16, activation='relu')(X)
#     X = Dropout(rate = 0.2)(X)
#     X = BatchNormalization()(X)
#     X = MaxPool1D(2, 2)(X)

#     X = Conv1D(kernel_size=13, filters=32, activation='relu')(X)
#     X = Dropout(rate = 0.2)(X)
#     X = BatchNormalization()(X)
#     X = MaxPool1D(2, 2)(X)

#     X = Conv1D(kernel_size=17, filters=64, activation='relu')(X)
#     X = Dropout(rate = 0.2)(X)
#     X = BatchNormalization()(X)
#     X = MaxPool1D(2, 2)(X)

#     X = Conv1D(kernel_size=21, filters=32, activation='relu')(X)
#     X = Dropout(rate = 0.2)(X)
#     X = BatchNormalization()(X)
#     X = MaxPool1D(2, 2)(X)

#     X = Conv1D(kernel_size=23, filters=32, activation='relu')(X)
#     X = Dropout(rate = 0.2)(X)
#     X = BatchNormalization()(X)
#     X = MaxPool1D(2, 2)(X)

#     X = Flatten()(X)

#     X = Dense(100, activation='relu')(X)
#     outputs = Dense(units = 1, name = 'sigmoid', activation='sigmoid')(X)

#     model = Model(inputs=inputs, outputs=outputs)

#     return model

## 7.5 seconds
# def build_model():

#     inputs = Input(shape=(int(t_batch * 256), 19))

#     X = tf.keras.layers.LayerNormalization()(inputs)

#     X = Conv1D(kernel_size=11, filters=16, activation='relu')(X)
#     X = Dropout(rate = 0.2)(X)
#     X = BatchNormalization()(X)
#     X = MaxPool1D(2, 2)(X)

#     X = Conv1D(kernel_size=15, filters=32, activation='relu')(X)
#     X = Dropout(rate = 0.2)(X)
#     X = BatchNormalization()(X)
#     X = MaxPool1D(2, 2)(X)

#     X = Conv1D(kernel_size=19, filters=64, activation='relu')(X)
#     X = Dropout(rate = 0.2)(X)
#     X = BatchNormalization()(X)
#     X = MaxPool1D(2, 2)(X)

#     X = Conv1D(kernel_size=23, filters=32, activation='relu')(X)
#     X = Dropout(rate = 0.2)(X)
#     X = BatchNormalization()(X)
#     X = MaxPool1D(2, 2)(X)

#     X = Conv1D(kernel_size=27, filters=32, activation='relu')(X)
#     X = Dropout(rate = 0.2)(X)
#     X = BatchNormalization()(X)
#     X = MaxPool1D(2, 2)(X)

#     X = Flatten()(X)

#     X = Dense(100, activation='relu')(X)
#     outputs = Dense(units = 1, name = 'sigmoid', activation='sigmoid')(X)

#     model = Model(inputs=inputs, outputs=outputs)

#     return model

## 10 seconds
# def build_model():

#     inputs = Input(shape=(int(t_batch * 256), 19))

#     X = tf.keras.layers.LayerNormalization()(inputs)

#     X = Conv1D(kernel_size=11, filters=16, activation='relu')(X)
#     X = Dropout(rate = 0.2)(X)
#     X = BatchNormalization()(X)
#     X = MaxPool1D(2, 2)(X)

#     X = Conv1D(kernel_size=15, filters=32, activation='relu')(X)
#     X = Dropout(rate = 0.2)(X)
#     X = BatchNormalization()(X)
#     X = MaxPool1D(2, 2)(X)

#     X = Conv1D(kernel_size=19, filters=64, activation='relu')(X)
#     X = Dropout(rate = 0.2)(X)
#     X = BatchNormalization()(X)
#     X = MaxPool1D(2, 2)(X)

#     X = Conv1D(kernel_size=23, filters=32, activation='relu')(X)
#     X = Dropout(rate = 0.2)(X)
#     X = BatchNormalization()(X)
#     X = MaxPool1D(2, 2)(X)

#     X = Conv1D(kernel_size=27, filters=32, activation='relu')(X)
#     X = Dropout(rate = 0.2)(X)
#     X = BatchNormalization()(X)
#     X = MaxPool1D(2, 2)(X)

#     X = Flatten()(X)

#     X = Dense(100, activation='relu')(X)
#     outputs = Dense(units = 1, name = 'sigmoid', activation='sigmoid')(X)

#     model = Model(inputs=inputs, outputs=outputs)

#     return model


## Single-channel analysis
# def build_model():

#     inputs = Input(shape=(int(t_batch * 256), 1))

#     X = Conv1D(kernel_size=11, filters=16, activation='relu')(inputs)
#     X = Dropout(rate = 0.2)(X)
#     X = BatchNormalization()(X)
#     X = MaxPool1D(2, 2)(X)

#     X = Conv1D(kernel_size=15, filters=32, activation='relu')(X)
#     X = Dropout(rate = 0.2)(X)
#     X = BatchNormalization()(X)
#     X = MaxPool1D(2, 2)(X)

#     X = Conv1D(kernel_size=19, filters=64, activation='relu')(X)
#     X = Dropout(rate = 0.2)(X)
#     X = BatchNormalization()(X)
#     X = MaxPool1D(2, 2)(X)

#     X = Conv1D(kernel_size=23, filters=32, activation='relu')(X)
#     X = Dropout(rate = 0.2)(X)
#     X = BatchNormalization()(X)
#     X = MaxPool1D(2, 2)(X)

#     X = Conv1D(kernel_size=27, filters=32, activation='relu')(X)
#     X = Dropout(rate = 0.2)(X)
#     X = BatchNormalization()(X)
#     X = MaxPool1D(2, 2)(X)

#     X = Flatten()(X)

#     X = Dense(100, activation='relu')(X)
#     outputs = Dense(units = 1, name = 'sigmoid', activation='sigmoid')(X)

#     model = Model(inputs=inputs, outputs=outputs)

#     return model

# Training Model

In [ ]:
k = 60               # number of folds
n = int (60 / k)     # number of persons per each fold

n_samples = int( (30 * 60) / (t_batch) )   # number of samples for each person

earlystopping = callbacks.EarlyStopping(monitor ="val_accuracy",
                                            mode ="max", patience = 5,
                                            restore_best_weights = True)

Accuracy = []
for i in range(0, 60):

    print('# Person :', i)

    X_train = np.concatenate((X[0 : i*n_samples*n], X[(i+1)*n_samples*n : ]), axis=0)
    Y_train = np.concatenate((Y[0 : i*n_samples*n], Y[(i+1)*n_samples*n : ]), axis=0)

    X_test  = X[i*n_samples*n : (i+1)*n_samples*n]
    Y_test  = Y[i*n_samples*n : (i+1)*n_samples*n]

    X_train, Y_train = shuffle(X_train, Y_train, random_state=21)

    print(X_train.shape)
    print(X_test.shape)
    print()

    model = build_model()
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(X_train, Y_train, validation_split=0.1, batch_size=64,
    epochs=20, callbacks=earlystopping)

    hist = model.evaluate(X_test, Y_test, verbose=0)
    Accuracy.append(hist[1])

    print(hist[1])
    print()
    model.save("/content/drive/MyDrive/Bachelor's Project/Models/2.5 Seconds/model" + str(i) + ".keras")

    del X_train, Y_train, X_test, Y_test
    del model
    tf.keras.backend.clear_session()


    # print(X_train.shape)
    # print(X_test.shape)
    # print()
    #callbacks=earlystopping

    # print('t :', t_batch)
    # print('# Person :', i)

    # case = 'Normal' if Y_test[0] == 0 else 'MCI'
    # print(case)


    # print(hist[1])
    # print()

    # del X_train, Y_train, X_test, Y_test
    # del model

    # for i in range(56, 60):

#     if Accuracy[i] < 0.75:

#         print(i)
#         count = 0

#         X_train = np.concatenate((X[0 : i*n_samples*n], X[(i+1)*n_samples*n : ]), axis=0)
#         Y_train = np.concatenate((Y[0 : i*n_samples*n], Y[(i+1)*n_samples*n : ]), axis=0)

#         X_test  = X[i*n_samples*n : (i+1)*n_samples*n]
#         Y_test  = Y[i*n_samples*n : (i+1)*n_samples*n]

#         X_train, Y_train = shuffle(X_train, Y_train, random_state=21)

#         while count != 2:

#             model = build_model()
#             model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
#             model.fit(X_train, Y_train, validation_split=0.1, batch_size=1024,
#             epochs=10, callbacks=earlystopping, verbose=0)

#             hist = model.evaluate(X_test, Y_test, verbose=0)

#             if hist[1] < Accuracy[i]:
#                 del model

#             elif hist[1] > Accuracy[i]:

#                 Accuracy[i] = hist[1]
#                 model.save("/content/drive/MyDrive/Bachelor's Project/Models/10 Seconds/model" + str(i) + ".keras")
#                 del model
#                 print('# Person :', i)
#                 print('Count :', count)
#                 print(Accuracy[i])
#                 print()

#             if hist[1] >= 0.75:
#                 break

#             count += 1

#         del X_train, Y_train, X_test, Y_test

# Saving Model

In [ ]:
# Every fold is already saved inside the training loop above.
# No additional save is required here.

# Evaluation & Results

In [ ]:
Accuracy = []

n_samples = int( (30 * 60) / (t_batch) )   # number of samples for each person

# i = 3

# model = tf.keras.models.load_model("/content/drive/MyDrive/Bachelor's Project/Models/2.5 Seconds/model" + str(i) + ".keras")
# X_test  = X[i*n_samples : (i+1)*n_samples]
# Y_test  = Y[i*n_samples : (i+1)*n_samples]
# pred = model.predict(X_test)
# hist = model.evaluate(X_test, Y_test, verbose=0)


Accuracy = np.zeros((60, ))
Precision = np.zeros((60, ))
Sensitivity = np.zeros((60, ))
Specificity = np.zeros((60, ))


def safe_divide(numerator, denominator):
    return numerator / denominator if denominator else np.nan


Y_true_all = []
Y_pred_all = []

for i in range(60):

    model = tf.keras.models.load_model("/content/drive/MyDrive/Bachelor's Project/Models/2.5 Seconds/model" + str(i) + ".keras")

    X_test  = X[i*n_samples : (i+1)*n_samples]
    Y_test  = Y[i*n_samples : (i+1)*n_samples]

    Y_pred = model.predict(X_test)
    Y_pred = (Y_pred.T >= 0.5)

    hist = model.evaluate(X_test, Y_test, verbose=0)

    TP = np.sum((Y_pred == True)  & (Y_pred == Y_test))
    TN = np.sum((Y_pred == False) & (Y_pred == Y_test))
    FP = np.sum((Y_pred == True)  & (Y_pred != Y_test))
    FN = np.sum((Y_pred == False) & (Y_pred != Y_test))

    Accuracy[i] = safe_divide(TP + TN, TP + TN + FP + FN)
    Precision[i] = safe_divide(TP, TP + FP)
    Sensitivity[i] = safe_divide(TP, TP + FN)
    Specificity[i] = safe_divide(TN, TN + FP)

    Y_true_all.append(Y_test.astype(int))
    Y_pred_all.append(Y_pred.reshape(-1).astype(int))

    print(i, hist[1], Accuracy[i], Precision[i], Sensitivity[i], Specificity[i])
    print()
    del model
    tf.keras.backend.clear_session()


Y_true_all = np.concatenate(Y_true_all)
Y_pred_all = np.concatenate(Y_pred_all)

TP = np.sum((Y_pred_all == 1) & (Y_true_all == 1))
TN = np.sum((Y_pred_all == 0) & (Y_true_all == 0))
FP = np.sum((Y_pred_all == 1) & (Y_true_all == 0))
FN = np.sum((Y_pred_all == 0) & (Y_true_all == 1))

pooled_metrics = {
    "accuracy": safe_divide(TP + TN, TP + TN + FP + FN),
    "precision": safe_divide(TP, TP + FP),
    "sensitivity": safe_divide(TP, TP + FN),
    "specificity": safe_divide(TN, TN + FP),
}

print("Pooled metrics:", pooled_metrics)

In [ ]:
try:
    import tkinter as tk
    from tkinter import ttk
    from tkinter import filedialog as fd
except ImportError:
    print("Tkinter is not installed in this Python environment.")
else:
    def func():
        selected_file = fd.askopenfilename(
            title="Select an EEG signal file",
            filetypes=(("MATLAB files", "*.mat"), ("All files", "*.*")),
        )
        if selected_file:
            print("Selected file:", selected_file)


    try:
        root = tk.Tk()
        root.title('برنامه تشخیص اختلال شناختی خفیف (MCI)')
        root.resizable(False, False)
        root.geometry('500x500')
        tk.Button(root, text="انتخاب فایل سیگنال", width=20, height=1, command=func).place(x=178, y=400)
        root.mainloop()
    except tk.TclError:
        print("Tkinter requires a local desktop display and cannot open inside Google Colab.")